# Notebook 00 - Vérification de l’environnement

Ce notebook sert à vérifier que le projet **OrderOps** utilise le bon environnement Python et que sa configuration est correctement chargée avant de poursuivre le développement.

Il permet de confirmer que :

- le notebook s’exécute bien depuis le projet **OrderOps** ;
- Python **3.12** est utilisé ;
- le kernel Jupyter utilise bien le `.venv` du projet ;
- les principales dépendances Python sont installées ;
- le fichier `.env` est correctement lu par `pydantic-settings` ;
- les paramètres PostgreSQL, MCP, Mistral et LangSmith sont accessibles ;
- les secrets sont bien configurés sans être affichés en clair ;
- les URLs de connexion sont correctement formatées.

Ce notebook ne teste pas encore les règles métier de l’application. Il constitue un **checkpoint d’environnement et de configuration** afin de détecter les erreurs techniques avant de commencer les tests fonctionnels.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

EXPECTED_VENV = PROJECT_ROOT / ".venv"

assert (PROJECT_ROOT / "pyproject.toml").is_file()
assert (PROJECT_ROOT / "src" / "orderops").is_dir()
assert sys.version_info[:2] == (3, 12)
assert Path(sys.prefix).resolve() == EXPECTED_VENV.resolve()

print("Projet :", PROJECT_ROOT)
print("Python :", sys.executable)
print("Version :", sys.version.split()[0])

In [ ]:
from importlib.metadata import version

PACKAGES = (
    "pydantic",
    "sqlalchemy",
    "psycopg",
    "langchain",
    "langgraph",
    "mcp",
    "fastapi",
)

for package in PACKAGES:
    print(f"{package}: {version(package)}")

In [ ]:
from sqlalchemy.engine import make_url

from orderops.config import get_settings

# Recharge la configuration depuis .env.
get_settings.cache_clear()
settings = get_settings()

# Prépare l'URL PostgreSQL sans afficher le mot de passe.
safe_database_url = make_url(settings.database_url).render_as_string(
    hide_password=True
)

print("Environnement :", settings.app_env)
print("Base SQLAlchemy :", safe_database_url)
print("Base checkpointer configurée :", bool(settings.checkpoint_database_url))
print("URL MCP :", settings.mcp_url)
print("Port MCP :", settings.mcp_port)
print("Modèle chat :", settings.mistral_chat_model)
print("Modèle embeddings :", settings.mistral_embed_model)
print("Clé Mistral renseignée :", bool(settings.mistral_api_key.get_secret_value()))
print("LangSmith tracing :", settings.langsmith_tracing)
print("Clé LangSmith renseignée :", bool(settings.langsmith_api_key.get_secret_value()))